# Lab 1 Practice : Structured Streaming Pipeline


📋 Vue d'ensemble
Objectif : Construire un pipeline Structured Streaming complet avec :

 .Source de streaming (fichiers JSON)

 . Agrégation windowed + watermark

 . Sink Parquet avec checkpoint

 . Monitoring et métriques

 . Optimisation avant/après

Résultat final : Dossier complet avec code, preuves et rapport d'optimisation

**Partie 1 : Définir le Schéma et Préparer le Répertoire de Source**

 **Track A (Esports)**

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
import time, pathlib, json

# Créer la session Spark
spark = SparkSession.builder \
    .appName("DE2-Lab1-Streaming") \
    .master("local[*]") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

# ============================================================
# PARTIE 1 : DÉFINIR LE SCHÉMA (Track A - Esports)
# ============================================================

# Schéma des événements Match
schema_esports = StructType([
    StructField("match_id", StringType(), False),
    StructField("match_end_time", TimestampType(), False),
    StructField("game_type", StringType(), True),
    StructField("winning_team", StringType(), True),
    StructField("match_duration_sec", IntegerType(), True),
    StructField("spectators", IntegerType(), True),
])

print("Schéma des événements Match (Track A):")
print(schema_esports)

# Créer le répertoire de source
landing_dir = "data/landing/lab1/"
pathlib.Path(landing_dir).mkdir(parents=True, exist_ok=True)
print(f"\n✅ Répertoire créé : {landing_dir}")

# Créer les répertoires de sortie
pathlib.Path("outputs/lab1/stream_sink").mkdir(parents=True, exist_ok=True)
pathlib.Path("outputs/lab1/checkpoint").mkdir(parents=True, exist_ok=True)
pathlib.Path("proof").mkdir(parents=True, exist_ok=True)
print("✅ Répertoires de sortie créés")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/23 10:34:03 WARN Utils: Your hostname, Wandaogo, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/23 10:34:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/23 10:34:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.1
Spark UI: http://10.255.255.254:4040
Schéma des événements Match (Track A):
StructType([StructField('match_id', StringType(), False), StructField('match_end_time', TimestampType(), False), StructField('game_type', StringType(), True), StructField('winning_team', StringType(), True), StructField('match_duration_sec', IntegerType(), True), StructField('spectators', IntegerType(), True)])

✅ Répertoire créé : data/landing/lab1/
✅ Répertoires de sortie créés


**Track B (Open-Source)**

In [2]:
schema_opensource = StructType([
    StructField("repo_id", StringType(), False),
    StructField("created_at", TimestampType(), False),
    StructField("event_type", StringType(), True),  # push, pull_request
    StructField("branch", StringType(), True),
    StructField("files_changed", IntegerType(), True),
])

**Track C (Micromobilité)**

In [3]:
schema_micromobility = StructType([
    StructField("trip_id", StringType(), False),
    StructField("starttime", TimestampType(), False),
    StructField("vehicle_type", StringType(), True),  # scooter, bike
    StructField("distance_km", DoubleType(), True),
    StructField("duration_min", IntegerType(), True),
])

**Track D (Aviation)**

In [4]:
schema_aviation = StructType([
    StructField("flight_id", StringType(), False),
    StructField("time_position", TimestampType(), False),
    StructField("aircraft_type", StringType(), True),
    StructField("altitude_ft", IntegerType(), True),
    StructField("speed_knots", IntegerType(), True),
])

**Partie 2 : Créer des Données de Test**

In [5]:
# ============================================================
# GÉNÉRER DES DONNÉES JSON DE TEST (Track A)
# ============================================================

import json
from datetime import datetime, timedelta
import random

def generate_esports_events(num_files=10, events_per_file=5):
    """Génère des fichiers JSON pour simuler un stream d'événements Match"""
    
    landing_dir = "data/landing/lab1/"
    base_time = datetime(2025, 5, 17, 10, 0, 0)
    
    game_types = ["FPS", "MOBA", "Strategy", "Fighting"]
    teams = ["Team A", "Team B", "Team C", "Team D", "Team E"]
    
    for file_idx in range(num_files):
        events = []
        for event_idx in range(events_per_file):
            # Incrémenter le temps
            event_time = base_time + timedelta(
                minutes=file_idx * 5 + event_idx * 2
            )
            
            event = {
                "match_id": f"match_{file_idx:03d}_{event_idx:02d}",
                "match_end_time": event_time.isoformat() + "Z",
                "game_type": random.choice(game_types),
                "winning_team": random.choice(teams),
                "match_duration_sec": random.randint(600, 3600),
                "spectators": random.randint(100, 50000),
            }
            events.append(event)
        
        # Écrire le fichier JSON
        filename = f"{landing_dir}events_{file_idx:03d}.json"
        with open(filename, "w") as f:
            for event in events:
                f.write(json.dumps(event) + "\n")
        
        print(f"✅ Créé : {filename}")

# Générer 10 fichiers de test
generate_esports_events(num_files=10, events_per_file=5)
print(f"\n✅ {10} fichiers JSON créés dans {landing_dir}")

✅ Créé : data/landing/lab1/events_000.json
✅ Créé : data/landing/lab1/events_001.json
✅ Créé : data/landing/lab1/events_002.json
✅ Créé : data/landing/lab1/events_003.json
✅ Créé : data/landing/lab1/events_004.json
✅ Créé : data/landing/lab1/events_005.json
✅ Créé : data/landing/lab1/events_006.json
✅ Créé : data/landing/lab1/events_007.json
✅ Créé : data/landing/lab1/events_008.json
✅ Créé : data/landing/lab1/events_009.json

✅ 10 fichiers JSON créés dans data/landing/lab1/


**Partie 3 : Créer la Source de Streaming**

In [6]:
# ============================================================
# PARTIE 3 : CRÉER LA SOURCE DE STREAMING
# ============================================================

landing_dir = "data/landing/lab1/"

# Créer le DataFrame de streaming
df_stream = (spark.readStream
    .schema(schema_esports)
    .option("header", False)
    .option("maxFilesPerTrigger", 1)  # 1 fichier par micro-batch
    .json(landing_dir))

print("Source de streaming créée")
print(f"Is streaming: {df_stream.isStreaming}")
df_stream.printSchema()

# Afficher un exemple d'événement (sans exécuter le stream)
print("\nÉxemple de structure:")
print("""
match_id: "match_000_00"
match_end_time: 2025-05-17T10:00:00Z
game_type: "FPS"
winning_team: "Team A"
match_duration_sec: 1500
spectators: 5000
""")

Source de streaming créée
Is streaming: True
root
 |-- match_id: string (nullable = true)
 |-- match_end_time: timestamp (nullable = true)
 |-- game_type: string (nullable = true)
 |-- winning_team: string (nullable = true)
 |-- match_duration_sec: integer (nullable = true)
 |-- spectators: integer (nullable = true)


Éxemple de structure:

match_id: "match_000_00"
match_end_time: 2025-05-17T10:00:00Z
game_type: "FPS"
winning_team: "Team A"
match_duration_sec: 1500
spectators: 5000



**Partie 4 : Appliquer Watermark + Windowed Aggregation**

In [7]:
# ============================================================
# PARTIE 4 : WATERMARK + WINDOWED AGGREGATION
# ============================================================

# Paramètres de streaming
EVENT_TIME_COL = "match_end_time"
WINDOW_DURATION = "1 hour"      # Track A : 1 heure
WATERMARK_DELAY = "10 minutes"  # Accepte les events jusqu'à 10 min en retard

# Appliquer le watermark et l'agrégation windowed
windowed = (df_stream
    .withWatermark(EVENT_TIME_COL, WATERMARK_DELAY)
    .groupBy(
        F.window(EVENT_TIME_COL, WINDOW_DURATION),
        F.col("game_type")
    )
    .agg(
        F.count("*").alias("nb_matches"),
        F.avg("match_duration_sec").alias("avg_duration_sec"),
        F.avg("spectators").alias("avg_spectators"),
        F.max("spectators").alias("max_spectators"),
        F.min("match_duration_sec").alias("min_duration_sec")
    )
    .select(
        "window.start",
        "window.end",
        "game_type",
        "nb_matches",
        "avg_duration_sec",
        "avg_spectators",
        "max_spectators",
        "min_duration_sec"
    ))

print("Schéma après agrégation windowed:")
windowed.printSchema()

print("\n=== Plan de la requête windowed ===")
windowed.explain("formatted")

Schéma après agrégation windowed:
root
 |-- start: timestamp (nullable = true)
 |-- end: timestamp (nullable = true)
 |-- game_type: string (nullable = true)
 |-- nb_matches: long (nullable = false)
 |-- avg_duration_sec: double (nullable = true)
 |-- avg_spectators: double (nullable = true)
 |-- max_spectators: integer (nullable = true)
 |-- min_duration_sec: integer (nullable = true)


=== Plan de la requête windowed ===
== Physical Plan ==
* HashAggregate (12)
+- StateStoreSave (11)
   +- * HashAggregate (10)
      +- StateStoreRestore (9)
         +- * HashAggregate (8)
            +- Exchange (7)
               +- * HashAggregate (6)
                  +- * Project (5)
                     +- * Filter (4)
                        +- EventTimeWatermark (3)
                           +- * Project (2)
                              +- StreamingRelation (1)


(1) StreamingRelation
Output [6]: [match_id#0, match_end_time#1, game_type#2, winning_team#3, match_duration_sec#4, spectators#5]


**Partie 5 : Écrire vers Parquet Sink**

In [8]:
# ============================================================
# PARTIE 5 : ÉCRIRE VERS PARQUET SINK
# ============================================================

# Lancer la requête de streaming
query = (windowed.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", "outputs/lab1/stream_sink")
    .option("checkpointLocation", "outputs/lab1/checkpoint")
    .trigger(processingTime="10 seconds")
    .start())

print(f"✅ Query lancée")
print(f"Query name: {query.name}")
print(f"Query id: {query.id}")
print(f"Status: {query.status}")

# Attendre 60 secondes pour laisser le stream traiter les données
print("\n⏳ Stream en cours... (60 secondes)")
query.awaitTermination(timeout=60)

# Arrêter la requête
query.stop()
print("\n✅ Query arrêtée")

26/04/23 10:38:25 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


✅ Query lancée
Query name: None
Query id: 2340ac8b-3188-4dca-88e1-adc3274bc457
Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}

⏳ Stream en cours... (60 secondes)



✅ Query arrêtée


26/04/23 10:39:29 WARN DAGScheduler: Failed to cancel job group 0c22854a-d950-4002-93fd-fb80f6c04023. Cannot find active jobs for it.
26/04/23 10:39:29 WARN DAGScheduler: Failed to cancel job group 0c22854a-d950-4002-93fd-fb80f6c04023. Cannot find active jobs for it.


**Partie 6 : Monitoring et Capture des Métriques**

In [10]:
# ============================================================
# PARTIE 6 : MONITORING ET CAPTURE DES MÉTRIQUES
# ============================================================

import json
from datetime import datetime

# Récupérer les métriques du dernier micro-batch
progress = query.lastProgress

if progress:
    print("=== Dernières Métriques ===\n")
    
    # Convertir progress en dictionnaire sérialisable
    progress_dict = dict(progress)
    
    # Fonction pour convertir les objets non-sérialisables
    def convert_to_serializable(obj):
        """Convertit les objets non-sérialisables en strings"""
        if isinstance(obj, dict):
            return {k: convert_to_serializable(v) for k, v in obj.items()}
        elif isinstance(obj, (list, tuple)):
            return [convert_to_serializable(item) for item in obj]
        elif isinstance(obj, (datetime, type(None))):
            return str(obj) if obj is not None else None
        elif hasattr(obj, '__dict__'):
            # Pour les objets custom, convertir en string
            return str(obj)
        else:
            return obj
    
    # Sérialiser le progress
    progress_serializable = convert_to_serializable(progress_dict)
    
    try:
        print(json.dumps(progress_serializable, indent=2))
    except Exception as e:
        print(f"Erreur lors de la sérialisation: {e}")
        print(progress_serializable)
    
    # Extraire les métriques clés
    print("\n=== Métriques Clés ===")
    print(f"Input Rows: {progress.get('numInputRows', 0)}")
    print(f"Output Rows: {progress.get('numOutputRows', 0)}")
    print(f"Batch Duration (ms): {progress.get('batchDuration', 0)}")
    print(f"Processing Delay (ms): {progress.get('processingDelay', 0)}")
    print(f"Input Rate (rows/sec): {progress.get('inputRowsPerSecond', 0):.2f}")
    print(f"Processing Rate (rows/sec): {progress.get('processedRowsPerSecond', 0):.2f}")
    
    # Sauvegarder le JSON complet
    with open("proof/query_progress.json", "w") as f:
        json.dump(progress_serializable, f, indent=2, default=str)
    print("\n✅ Métriques sauvegardées dans proof/query_progress.json")
else:
    print("⚠️ Aucune métrique disponible - la requête n'a pas traité de données")

# Sauvegarder le plan d'exécution
print("\n=== Plan de la requête windowed ===")
windowed.explain("formatted")

# Capturer le plan dans un fichier (via redirection)
with open("proof/plan_streaming.txt", "w") as f:
    f.write("=== PLAN D'EXÉCUTION DU STREAMING ===\n\n")
    f.write("Schéma de sortie:\n")
    f.write(str(windowed.schema) + "\n\n")
    f.write("Note: Le plan détaillé est affiché dans la sortie console ci-dessus.\n")
    f.write("Utilisez windowed.explain('formatted') pour voir le plan complet.\n")

print("\n✅ Plan sauvegardé dans proof/plan_streaming.txt")

=== Dernières Métriques ===

Erreur lors de la sérialisation: Object of type UUID is not JSON serializable
{'id': UUID('2340ac8b-3188-4dca-88e1-adc3274bc457'), 'runId': UUID('0c22854a-d950-4002-93fd-fb80f6c04023'), 'name': None, 'timestamp': '2026-04-23T08:39:20.000Z', 'batchId': 6, 'batchDuration': 2441, 'durationMs': {'triggerExecution': 2441, 'queryPlanning': 16, 'getBatch': 6, 'commitOffsets': 21, 'latestOffset': 20, 'addBatch': 2354, 'walCommit': 19}, 'eventTime': {'min': '2025-05-17T10:15:00.000Z', 'avg': '2025-05-17T10:19:00.000Z', 'watermark': '2025-05-17T10:38:00.000Z', 'max': '2025-05-17T10:23:00.000Z'}, 'stateOperators': [{'operatorName': 'stateStoreSave', 'numRowsTotal': 4, 'numRowsUpdated': 4, 'numRowsRemoved': 0, 'allUpdatesTimeMs': 346, 'allRemovalsTimeMs': 456, 'commitTimeMs': 16826, 'memoryUsedBytes': 91264, 'numRowsDroppedByWatermark': 0, 'numShufflePartitions': 200, 'numStateStoreInstances': 200, 'customMetrics': {'stateOnCurrentVersionSizeBytes': 25056, 'loadedMapCa

**Partie 7 : Vérifier les Outputs**

In [11]:
# ============================================================
# VÉRIFIER LES FICHIERS GÉNÉRÉS
# ============================================================

import os

print("=== Vérification des Outputs ===\n")

# Vérifier les fichiers Parquet
sink_dir = "outputs/lab1/stream_sink"
if os.path.exists(sink_dir):
    files = os.listdir(sink_dir)
    print(f"📁 {sink_dir}/ contient {len(files)} fichiers:")
    for f in files[:5]:
        print(f"   - {f}")

# Vérifier le checkpoint
checkpoint_dir = "outputs/lab1/checkpoint"
if os.path.exists(checkpoint_dir):
    files = os.listdir(checkpoint_dir)
    print(f"\n📁 {checkpoint_dir}/ contient {len(files)} fichiers")

# Vérifier les preuves
proof_dir = "proof"
if os.path.exists(proof_dir):
    files = os.listdir(proof_dir)
    print(f"\n📁 {proof_dir}/ contient:")
    for f in files:
        print(f"   - {f}")

=== Vérification des Outputs ===

📁 outputs/lab1/stream_sink/ contient 15 fichiers:
   - part-00000-6d6737fa-f164-4be1-9876-d43549bf2d3d-c000.snappy.parquet
   - .part-00000-df3d43d2-e3dc-4e14-a03e-704207558029-c000.snappy.parquet.crc
   - part-00000-7076a30e-a1c6-43e5-af8d-c28ba50091a2-c000.snappy.parquet
   - part-00000-df3d43d2-e3dc-4e14-a03e-704207558029-c000.snappy.parquet
   - _spark_metadata

📁 outputs/lab1/checkpoint/ contient 6 fichiers

📁 proof/ contient:
   - query_progress.json
   - plan_streaming.txt


**Partie 8 : Optimisation (Before/After)**

In [12]:
# ============================================================
# PARTIE 8 : OPTIMISATION ET RE-MESURE
# ============================================================

# Configuration OPTIMISÉE
WINDOW_DURATION_OPT = "30 minutes"  # Plus court (latence ⬇️)
WATERMARK_DELAY_OPT = "5 minutes"   # Plus strict (mémoire ⬇️)

# Créer la source optimisée
df_stream_opt = (spark.readStream
    .schema(schema_esports)
    .option("header", False)
    .option("maxFilesPerTrigger", 1)
    .json(landing_dir))

# Windowed aggregation optimisée
windowed_opt = (df_stream_opt
    .withWatermark(EVENT_TIME_COL, WATERMARK_DELAY_OPT)
    .groupBy(
        F.window(EVENT_TIME_COL, WINDOW_DURATION_OPT),
        F.col("game_type")
    )
    .agg(
        F.count("*").alias("nb_matches"),
        F.avg("match_duration_sec").alias("avg_duration_sec"),
        F.avg("spectators").alias("avg_spectators")
    ))

# Lancer la requête optimisée
query_opt = (windowed_opt.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", "outputs/lab1/stream_sink_optimized")
    .option("checkpointLocation", "outputs/lab1/checkpoint_optimized")
    .trigger(processingTime="5 seconds")  # Plus rapide
    .start())

print("✅ Query optimisée lancée")
query_opt.awaitTermination(timeout=60)
query_opt.stop()

# Capturer les métriques optimisées
progress_opt = query_opt.lastProgress

print("\n=== Métriques Optimisées ===")
if progress_opt:
    print(f"Input Rows: {progress_opt.get('numInputRows', 0)}")
    print(f"Processing Delay (ms): {progress_opt.get('processingDelay', 0)}")
    print(f"Processing Rate (rows/sec): {progress_opt.get('processedRowsPerSecond', 0):.2f}")

26/04/23 10:42:07 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


✅ Query optimisée lancée



=== Métriques Optimisées ===
Input Rows: 0
Processing Delay (ms): 0
Processing Rate (rows/sec): 0.00


26/04/23 10:43:09 WARN DAGScheduler: Failed to cancel job group dff89fb7-f687-462c-b7fa-73a92fe895f5. Cannot find active jobs for it.
26/04/23 10:43:09 WARN DAGScheduler: Failed to cancel job group dff89fb7-f687-462c-b7fa-73a92fe895f5. Cannot find active jobs for it.


**Partie 9 : Créer le Log des Métriques**

In [13]:
# ============================================================
# PARTIE 9 : CRÉER LE LOG DES MÉTRIQUES (CSV)
# ============================================================

import pandas as pd

# Construire le log
metrics_data = {
    "run_id": ["baseline", "optimized"],
    "trigger_interval_sec": [10, 5],
    "watermark_delay_min": [10, 5],
    "window_duration": ["1 hour", "30 minutes"],
    "input_rows": [
        progress.get('numInputRows', 0) if progress else 0,
        progress_opt.get('numInputRows', 0) if progress_opt else 0
    ],
    "output_rows": [
        progress.get('numOutputRows', 0) if progress else 0,
        progress_opt.get('numOutputRows', 0) if progress_opt else 0
    ],
    "batch_duration_ms": [
        progress.get('batchDuration', 0) if progress else 0,
        progress_opt.get('batchDuration', 0) if progress_opt else 0
    ],
    "processing_delay_ms": [
        progress.get('processingDelay', 0) if progress else 0,
        progress_opt.get('processingDelay', 0) if progress_opt else 0
    ],
    "input_rate_rows_sec": [
        progress.get('inputRowsPerSecond', 0) if progress else 0,
        progress_opt.get('inputRowsPerSecond', 0) if progress_opt else 0
    ],
    "processing_rate_rows_sec": [
        progress.get('processedRowsPerSecond', 0) if progress else 0,
        progress_opt.get('processedRowsPerSecond', 0) if progress_opt else 0
    ],
}

metrics_df = pd.DataFrame(metrics_data)

# Sauvegarder en CSV
metrics_df.to_csv("lab1_metrics_log.csv", index=False)

print("✅ Métriques sauvegardées dans lab1_metrics_log.csv\n")
print(metrics_df)

✅ Métriques sauvegardées dans lab1_metrics_log.csv

      run_id  trigger_interval_sec  watermark_delay_min window_duration  \
0   baseline                    10                   10          1 hour   
1  optimized                     5                    5      30 minutes   

   input_rows  output_rows  batch_duration_ms  processing_delay_ms  \
0           5            0               2441                    0   
1           0            0               2246                    0   

   input_rate_rows_sec  processing_rate_rows_sec  
0              0.50005                  2.048341  
1              0.00000                  0.000000  


**Partie 10 : Cleanup**

In [ ]:
# ============================================================
# PARTIE 8 : CLEANUP
# ============================================================

# Arrêter la session Spark
spark.stop()

print("\n" + "="*60)
print("✅ Lab 1 Practice COMPLET")
print("="*60)
print("\nFichiers générés:")
print("  📁 outputs/lab1/stream_sink/ → Parquet data")
print("  📁 outputs/lab1/checkpoint/ → Checkpoint files")
print("  📁 proof/ → Evidence (plan, screenshots)")
print("  📊 lab1_metrics_log.csv → Metrics comparison")
print("\nProchaines étapes:")
print("  1. Prendre des screenshots du Streaming UI")
print("  2. Rédiger le rapport d'optimisation (1 page)")
print("  3. Sauvegarder sur GitHub")


✅ Lab 1 Practice COMPLET

Fichiers générés:
  📁 outputs/lab1/stream_sink/ → Parquet data
  📁 outputs/lab1/checkpoint/ → Checkpoint files
  📁 proof/ → Evidence (plan, screenshots)
  📊 lab1_metrics_log.csv → Metrics comparison

Prochaines étapes:
  1. Prendre des screenshots du Streaming UI
  2. Rédiger le rapport d'optimisation (1 page)
  3. Sauvegarder sur GitHub


26/04/23 10:45:51 WARN StateStore: Error running maintenance thread
java.lang.IllegalStateException: SparkEnv not active, cannot do maintenance on StateStores
	at org.apache.spark.sql.execution.streaming.state.StateStore$.doMaintenance(StateStore.scala:971)
	at org.apache.spark.sql.execution.streaming.state.StateStore$.$anonfun$startMaintenanceIfNeeded$1(StateStore.scala:945)
	at org.apache.spark.sql.execution.streaming.state.StateStore$MaintenanceTask$$anon$1.run(StateStore.scala:746)
	at java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:572)
	at java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:358)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1144)
	at java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:642)
	at java.base/java.lang.Thread.